# DriverlessCars — Google Colab (GPU)

1. **Runtime → Change runtime type → Hardware accelerator → GPU** (then Save).
2. **Run all cells in order** (Runtime → Run all) — do not skip the clone cell.
3. The last cell prints a **gradio.live** public URL — open it to use the demo.

**Reopened the notebook but UI looks old?** Closing/reopening does **not** update `/content/DriverlessCars`. After any GitHub push, either:
- **Runtime → Restart session**, then run **all** cells from the top, **or**
- Re-run only the **clone** cell (deletes old code) and then install + launch.

**Fresh notebook from GitHub:** [Open in Colab](https://colab.research.google.com/github/ShahramChaudhry/DriverlessCars/blob/claude/autonomous-driving-demo-Ljmeg/colab/DriverlessCars_Colab.ipynb) (avoids an old copy saved in your Google Drive).

**Get the code on Colab** (pick one):
- **Public GitHub:** run the clone cell below. If you forked the repo, edit `REPO_URL` there.
- **If `git clone` fails with exit 128:** confirm the repo is **Public** on GitHub.
- **If stderr says `Unable to read current working directory`:** **Runtime → Restart session**, run the first code cell (`os.chdir("/content")`), then clone again.
- **Private / no GitHub:** upload a zip to `/content/DriverlessCars` and skip clone.

In [ ]:
import os

os.chdir("/content")

!nvidia-smi
import torch
print("cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))

### Clone / refresh code from GitHub
**Run this every time** you want the latest UI and fixes (it replaces `/content/DriverlessCars`).

In [ ]:
%cd /content

REPO_URL = "https://github.com/ShahramChaudhry/DriverlessCars.git"
BRANCH = "claude/autonomous-driving-demo-Ljmeg"

!rm -rf /content/DriverlessCars
!git clone --depth 1 --branch {BRANCH} {REPO_URL} /content/DriverlessCars

%cd /content/DriverlessCars
!git log -1 --oneline
print("Clone OK — commit above should match latest on GitHub")

### Install dependencies
**Use `requirements-colab.txt`** — it skips `torch` so Colab keeps its CUDA PyTorch.  
`pip install -r requirements.txt` replaces GPU torch with CPU-only (~2 FPS benchmarks, AMP useless).

In [ ]:
%cd /content/DriverlessCars
!pip install -q -r requirements-colab.txt

import torch
print("cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))
else:
    print("WARNING: CUDA off — Runtime → Change runtime type → GPU, then restart from top.")

### Launch Gradio (public link)
Wait until you see a **gradio.app** / **trycloudflare.com** URL, then click it. Keep this cell running.

Re-running this cell **fetches the latest GitHub commit**, reloads all Python modules, closes the old server, and starts fresh — no full re-clone needed.

In [ ]:
%cd /content/DriverlessCars
import importlib
import os
import py_compile
import subprocess
import sys

REPO = "/content/DriverlessCars"
BRANCH = "claude/autonomous-driving-demo-Ljmeg"
PY_FILES = ("config.py", "model_loader.py", "inference.py", "benchmark.py", "app.py")
RELOAD_ORDER = ("config", "model_loader", "pruning", "inference", "benchmark", "app")


def git_head() -> str:
    r = subprocess.run(
        ["git", "-C", REPO, "log", "-1", "--oneline"],
        capture_output=True,
        text=True,
    )
    return r.stdout.strip() if r.returncode == 0 else ""


def refresh_from_github() -> None:
    if not os.path.isdir(os.path.join(REPO, ".git")):
        print("No git repo — re-run the clone cell above.")
        return
    print("Before:", git_head() or "(unknown)")
    fetch = subprocess.run(
        ["git", "-C", REPO, "fetch", "origin", BRANCH, "--depth", "1"],
        capture_output=True,
        text=True,
    )
    if fetch.returncode != 0:
        print("git fetch failed — re-run the clone cell.\n", fetch.stderr.strip())
        return
    reset = subprocess.run(
        ["git", "-C", REPO, "reset", "--hard", f"origin/{BRANCH}"],
        capture_output=True,
        text=True,
    )
    if reset.returncode != 0:
        print("git reset failed:\n", reset.stderr.strip())
        return
    print("After:", git_head())


def compile_project() -> None:
    for name in PY_FILES:
        path = os.path.join(REPO, name)
        py_compile.compile(path, doraise=True)
    print("Syntax OK:", ", ".join(PY_FILES))


def reload_project_modules():
    """Reload all local modules so inference/config changes apply without restart."""
    if REPO not in sys.path:
        sys.path.insert(0, REPO)
    for name in RELOAD_ORDER:
        if name in sys.modules:
            importlib.reload(sys.modules[name])
        else:
            importlib.import_module(name)
    return sys.modules["app"]


def close_previous_demo() -> None:
    old_app = sys.modules.get("app")
    if old_app is None:
        return
    demo = getattr(old_app, "demo", None)
    if demo is None:
        return
    try:
        demo.close()
        print("Closed previous Gradio server.")
    except Exception as exc:
        print(f"Note: could not close previous server ({exc}). Restart runtime if the link is stale.")


refresh_from_github()
compile_project()
close_previous_demo()
app = reload_project_modules()
app.launch_gradio(share=True)